# E0-T04 — locked Colab GPU smoke test
Run top-to-bottom in a **fresh GPU runtime**. This notebook installs the committed lock, captures the runtime, loads only one model at a time, and writes non-sensitive pass/fail manifests to Drive. A Hugging Face token is optional because the pinned models are public.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Before continuing, add a My Drive shortcut to the shared `CMPE253-Project` folder if it is not already visible. Do not paste tokens or private paths into committed files.

In [ ]:
import os
from pathlib import Path
from datetime import UTC, datetime
from uuid import uuid4

REPO = Path("/content/cmpe253-llm-demographic-bias-audit")
drive_root = Path("/content/drive").resolve()
default_shared = drive_root / "MyDrive" / "CMPE253-Project"
if default_shared.is_dir():
    shared_root = str(default_shared)
else:
    my_drive = drive_root / "MyDrive"
    if my_drive.is_dir():
        print("MyDrive folders:", sorted(p.name for p in my_drive.iterdir() if p.is_dir()))
    shared_root = os.environ.get("AUDIT_SHARED_ROOT") or input(
        "Mounted project folder path from Colab Files pane (not a browser URL): "
    ).strip()
    if "://" in shared_root:
        raise ValueError("Paste a Colab filesystem path, not the Drive web URL")
SHARED = Path(shared_root).expanduser().resolve()
ACTOR = "vineet"
if not SHARED.is_dir() or not SHARED.is_relative_to(drive_root):
    raise ValueError("Project folder must exist under /content/drive; inspect the Colab Files pane")
if not (SHARED / "manifests").is_dir():
    raise ValueError("This is not the initialized project folder: manifests/ is missing")
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN_DIR = SHARED / "manifests" / "e0_t04_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Drive run directory:", RUN_DIR.relative_to(SHARED))

In [ ]:
!if [ -d "{REPO}/.git" ]; then git -C "{REPO}" pull --ff-only; else git clone https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git "{REPO}"; fi  # noqa: E501
%cd {REPO}
!git rev-parse HEAD

In [ ]:
!pip -q install uv
!uv sync --frozen --group dev --group colab

In [ ]:
!nvidia-smi
!PYTHONPATH=src uv run --frozen --group colab python -m environment_tools capture --output "{RUN_DIR}/e0_t04_{ACTOR}_environment.json"  # noqa: E501

## M1: Qwen
This process exits after the run, so its model memory is released before M2 starts. The test performs two passes over 20 prompts plus repeated likelihood scoring and fails on any tolerance violation.

In [ ]:
!PYTHONPATH=src uv run --frozen --group colab python -m inference.smoke_test --model-key M1 --output "{RUN_DIR}/e0_t04_{ACTOR}_M1_smoke.json"  # noqa: E501

## M2: Phi

In [ ]:
!PYTHONPATH=src uv run --frozen --group colab python -m inference.smoke_test --model-key M2 --output "{RUN_DIR}/e0_t04_{ACTOR}_M2_smoke.json"  # noqa: E501

In [ ]:
import json

for key in ("M1", "M2"):
    path = RUN_DIR / f"e0_t04_{ACTOR}_{key}_smoke.json"
    result = json.loads(path.read_text())
    assert result["status"] == "pass", result
    print(key, result["status"], result["fixture"])